# Finetune LLM
This pipeline performs the finetuning of an LLM, outputting the checkpoint when training ends

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, TaskType
import os

In [ ]:
# from datasets import Features, Value, ClassLabel, Sequence

# label2id = {
#     "O": 0,
#     "B-ORG": 1,
#     "I-ORG": 2,
#     "B-PER": 3,
#     "I-PER": 4,
#     "B-LOC": 5,
#     "I-LOC": 6,
#     "B-MISC": 7,
#     "I-MISC": 8,
# }

# features = Features(
#     {
#         "id": Value(dtype="string", id=None),
#         "tokens": Sequence(feature=Value(dtype="string", id=None), id=None),
#         "ner_tags": Sequence(ClassLabel(names=list(label2id.keys()))),
#     }
# )

# ds = load_dataset(
#     "json",
#     data_files="../data/labelled/all_hf.jsonl",
#     features=features,
#     split="train",
# )


# def preprocess(sample):
#     prompt = " ".join(sample["tokens"])
#     return {"text": prompt}


# dataset = ds.train_test_split(test_size=0.3)
# dataset = dataset.map(preprocess, remove_columns=["id", "tokens", "ner_tags"])
# dataset

In [ ]:
from datasets import Features, Value, ClassLabel, Sequence


ds = load_dataset(
    "json",
    data_files="../data/labelled/all_parallel.jsonl",
    split="train",
)


dataset = ds.train_test_split(test_size=0.3)
dataset

In [ ]:
from transformers import BitsAndBytesConfig

# model_name = "deepseek-ai/deepseek-llm-7b-base"
model_name = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
)

In [ ]:
rank_dimension = 16
lora_alpha = 64
lora_dropout = 0.1

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type=TaskType.CAUSAL_LM,
)

In [ ]:
output_dir = f"../models/{model_name}_legal_parallel"

per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4
learning_rate = 1e-4

num_train_epochs = 10
warmup_ratio = 0.1
lr_scheduler_type = "cosine"
max_seq_length = 1500

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_strategy="no",
    eval_strategy="epoch",
    learning_rate=learning_rate,
    weight_decay=0.1,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    bf16=True,
    hub_private_repo=False,
    push_to_hub=False,
    num_train_epochs=num_train_epochs,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    packing=False,
    max_seq_length=max_seq_length,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

In [ ]:
trainer.save_model(output_dir)

In [ ]:
# Ensure the directory exists (good practice)
if not os.path.isdir(output_dir):
    raise FileNotFoundError(f"Local checkpoint directory not found: {output_dir}")

local_model = AutoModelForCausalLM.from_pretrained(output_dir)